# ĐỒ ÁN KHAI PHÁ DỮ LIỆU
## Dự đoán Mức độ Hài lòng của Khách hàng

**Quy trình CRISP-DM**

---
**Sinh viên**: [Tên sinh viên]  
**Lớp**: [Lớp]  
**Giảng viên hướng dẫn**: [Tên giảng viên]

---

## Mục lục
1. [Phase 1: Business Understanding](#phase1)
2. [Phase 2: Data Understanding](#phase2)
3. [Phase 3: Data Preparation](#phase3)
4. [Phase 4: Modeling](#phase4)
5. [Phase 5: Evaluation](#phase5)
6. [Phase 6: Deployment](#phase6)


<a id="phase1"></a>
# PHASE 1: BUSINESS UNDERSTANDING

## 1.1. Mô tả Bài toán

**Vấn đề thực tế:**
- Khách hàng không hài lòng thường rời bỏ dịch vụ (churn), gây thiệt hại về doanh thu
- Việc đo lường mức độ hài lòng thường được thực hiện sau khi khách hàng đã sử dụng dịch vụ
- Cần một công cụ để dự đoán trước mức độ hài lòng

**Mục tiêu:**
Xây dựng mô hình Machine Learning để dự đoán mức độ hài lòng của khách hàng dựa trên các đặc trưng dịch vụ và thông tin khách hàng.

**Loại bài toán:** Binary Classification (Hài lòng / Không hài lòng)


In [ ]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, 
                            roc_auc_score, accuracy_score, precision_score, 
                            recall_score, f1_score)
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Đã import các thư viện thành công!")
print("📊 Sẵn sàng bắt đầu quy trình CRISP-DM")


<a id="phase2"></a>
# PHASE 2: DATA UNDERSTANDING

## 2.1. Thu thập Dữ liệu

**Nguồn dữ liệu:**
- Dataset tham khảo từ Kaggle: "Predicting Customer Satisfaction"
- Link: https://www.kaggle.com/code/andresionek/predicting-customer-satisfaction/

## 2.2. Load và Khảo sát Dữ liệu


In [ ]:
# Thử load dataset từ Kaggle, nếu không có thì tạo dữ liệu mẫu
dataset_paths = [
    'data/customer_satisfaction.csv',
    'data/train.csv',
    'customer_satisfaction.csv',
    'train.csv'
]

df = None
for path in dataset_paths:
    if os.path.exists(path):
        print(f"✅ Đang load dataset từ: {path}")
        try:
            df = pd.read_csv(path)
            print(f"✅ Đã load dataset thành công từ Kaggle!")
            print(f"   Số lượng mẫu: {len(df)}")
            print(f"   Số lượng cột: {len(df.columns)}")
            break
        except Exception as e:
            print(f"⚠️  Lỗi khi load {path}: {e}")
            continue

# Nếu không load được dataset, tạo dữ liệu mẫu
if df is None:
    print("⚠️  Không tìm thấy dataset từ Kaggle")
    print("   Đang tạo dữ liệu mẫu để demo...")
    print()
    
    np.random.seed(42)
    n_samples = 1000
    
    data = {
        'Age': np.random.randint(18, 80, n_samples),
        'Flight_Distance': np.random.randint(100, 5000, n_samples),
        'Inflight_wifi_service': np.random.randint(0, 6, n_samples),
        'Departure_Arrival_time_convenient': np.random.randint(0, 6, n_samples),
        'Ease_of_Online_booking': np.random.randint(0, 6, n_samples),
        'Gate_location': np.random.randint(0, 6, n_samples),
        'Food_and_drink': np.random.randint(0, 6, n_samples),
        'Online_boarding': np.random.randint(0, 6, n_samples),
        'Seat_comfort': np.random.randint(0, 6, n_samples),
        'Inflight_entertainment': np.random.randint(0, 6, n_samples),
        'On_board_service': np.random.randint(0, 6, n_samples),
        'Leg_room_service': np.random.randint(0, 6, n_samples),
        'Baggage_handling': np.random.randint(0, 6, n_samples),
        'Checkin_service': np.random.randint(0, 6, n_samples),
        'Inflight_service': np.random.randint(0, 6, n_samples),
        'Cleanliness': np.random.randint(0, 6, n_samples),
        'Gender': np.random.choice(['Male', 'Female'], n_samples),
        'Customer_Type': np.random.choice(['Loyal Customer', 'disloyal Customer'], n_samples),
        'Type_of_Travel': np.random.choice(['Business travel', 'Personal Travel'], n_samples),
        'Class': np.random.choice(['Business', 'Eco', 'Eco Plus'], n_samples, p=[0.2, 0.6, 0.2]),
    }
    
    df = pd.DataFrame(data)
    
    # Tạo target variable
    service_scores = df[['Inflight_wifi_service', 'Food_and_drink', 'Seat_comfort', 
                         'Inflight_entertainment', 'On_board_service', 'Inflight_service']].sum(axis=1)
    df['satisfaction'] = ((service_scores > 20) & 
                          (df['Customer_Type'] == 'Loyal Customer') & 
                          (df['Class'].isin(['Business', 'Eco Plus']))).astype(int)
    
    # Thêm noise
    noise = np.random.random(n_samples) < 0.15
    df.loc[noise, 'satisfaction'] = 1 - df.loc[noise, 'satisfaction']

print(f"\n📊 Thông tin Dataset:")
print(f"   Số lượng mẫu: {len(df)}")
print(f"   Số lượng features: {len(df.columns) - 1}")
print(f"   Số lượng cột: {len(df.columns)}")


In [ ]:
# Hiển thị thông tin cơ bản về dataset
print("=" * 60)
print("THÔNG TIN DATASET")
print("=" * 60)
df.info()


In [ ]:
# Hiển thị 5 dòng đầu tiên
print("=" * 60)
print("5 DÒNG ĐẦU TIÊN")
print("=" * 60)
df.head()


In [ ]:
# Thống kê mô tả
print("=" * 60)
print("THỐNG KÊ MÔ TẢ")
print("=" * 60)
df.describe()


In [ ]:
# Phân bố target variable
print("=" * 60)
print("PHÂN BỐ TARGET VARIABLE")
print("=" * 60)
target_dist = df['satisfaction'].value_counts()
print(target_dist)
print(f"\nTỷ lệ:")
print(f"  Không hài lòng (0): {target_dist[0]/len(df)*100:.2f}%")
print(f"  Hài lòng (1): {target_dist[1]/len(df)*100:.2f}%")

# Vẽ biểu đồ
plt.figure(figsize=(8, 6))
target_dist.plot(kind='bar', color=['#ef4444', '#10b981'])
plt.title('Phân bố Mức độ Hài lòng', fontsize=14, fontweight='bold')
plt.xlabel('Satisfaction', fontsize=12)
plt.ylabel('Số lượng', fontsize=12)
plt.xticks([0, 1], ['Không hài lòng', 'Hài lòng'], rotation=0)
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(target_dist):
    plt.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


<a id="phase3"></a>
# PHASE 3: DATA PREPARATION

## 3.1. Tách Features và Target

## 3.2. Xử lý Categorical Variables (One-Hot Encoding)

## 3.3. Train/Test Split

## 3.4. Feature Scaling


In [ ]:
# Tách features và target
X = df.drop('satisfaction', axis=1)
y = df['satisfaction']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")


In [ ]:
# One-hot encoding cho categorical variables
X_encoded = pd.get_dummies(X, columns=['Gender', 'Customer_Type', 'Type_of_Travel', 'Class'], 
                          drop_first=True)

print(f"Số features trước encoding: {X.shape[1]}")
print(f"Số features sau encoding: {X_encoded.shape[1]}")
print(f"\nCác features sau encoding:")
print(X_encoded.columns.tolist())

# Lưu tên features
feature_names = X_encoded.columns.tolist()


In [ ]:
# Kiểm tra missing values
print("=" * 60)
print("KIỂM TRA MISSING VALUES")
print("=" * 60)
missing = X_encoded.isnull().sum()
if missing.sum() == 0:
    print("✅ Không có missing values")
else:
    print("⚠️  Có missing values:")
    print(missing[missing > 0])


In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Giữ tỷ lệ phân bố lớp
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nPhân bố target trong train set:")
print(y_train.value_counts())
print(f"\nPhân bố target trong test set:")
print(y_test.value_counts())


In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Đã chuẩn hóa dữ liệu")
print(f"Train scaled shape: {X_train_scaled.shape}")
print(f"Test scaled shape: {X_test_scaled.shape}")


<a id="phase4"></a>
# PHASE 4: MODELING

## 4.1. Lựa chọn Mô hình

Chúng ta sẽ thử nghiệm 3 mô hình:
1. **Logistic Regression** - Mô hình tuyến tính baseline
2. **Random Forest** - Ensemble method với nhiều decision trees
3. **Gradient Boosting** - Boosting algorithm

## 4.2. Huấn luyện và Đánh giá Mô hình


In [ ]:
# Khởi tạo các mô hình
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

print("✅ Đã khởi tạo 3 mô hình:")
for name in models.keys():
    print(f"  - {name}")


In [ ]:
# Huấn luyện và đánh giá từng mô hình
results = {}

print("=" * 60)
print("HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH")
print("=" * 60)

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Mô hình: {name}")
    print(f"{'='*60}")
    
    # Huấn luyện
    model.fit(X_train_scaled, y_train)
    print("✅ Đã huấn luyện xong")
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"📊 Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    
    # Dự đoán trên test set
    y_pred = model.predict(X_test_scaled)
    
    # Tính metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }
    
    print(f"📈 Test Metrics:")
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   ROC-AUC:   {auc:.4f}")


In [ ]:
# So sánh các mô hình
print("=" * 60)
print("SO SÁNH CÁC MÔ HÌNH")
print("=" * 60)

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1-Score': [results[m]['f1'] for m in results.keys()],
    'ROC-AUC': [results[m]['auc'] for m in results.keys()]
})

comparison_df = comparison_df.sort_values('F1-Score', ascending=False)
print(comparison_df.to_string(index=False))

# Chọn mô hình tốt nhất (dựa trên F1-Score)
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f"\n🏆 Mô hình tốt nhất: {best_model_name}")
print(f"   F1-Score: {results[best_model_name]['f1']:.4f}")
print(f"   Accuracy: {results[best_model_name]['accuracy']:.4f}")


<a id="phase5"></a>
# PHASE 5: EVALUATION

## 5.1. Đánh giá Mô hình Tốt nhất

## 5.2. Confusion Matrix

## 5.3. Feature Importance


In [ ]:
# Đánh giá mô hình tốt nhất
y_pred_best = best_model.predict(X_test_scaled)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 60)
print(f"ĐÁNH GIÁ MÔ HÌNH: {best_model_name}")
print("=" * 60)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_best))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
print(cm)

# Vẽ confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Không hài lòng', 'Hài lòng'],
            yticklabels=['Không hài lòng', 'Hài lòng'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance (nếu mô hình hỗ trợ)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("=" * 60)
    print("TOP 10 FEATURES QUAN TRỌNG NHẤT")
    print("=" * 60)
    print(feature_importance.head(10).to_string(index=False))
    
    # Vẽ biểu đồ
    plt.figure(figsize=(10, 8))
    top_features = feature_importance.head(15)
    plt.barh(range(len(top_features)), top_features['importance'], color='#4f46e5')
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance', fontsize=12)
    plt.title('Top 15 Feature Importance', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Mô hình này không hỗ trợ feature importance")


<a id="phase6"></a>
# PHASE 6: DEPLOYMENT

## 6.1. Lưu Mô hình

## 6.2. Tóm tắt Kết quả

## 6.3. Ứng dụng Web

Để sử dụng mô hình trong thực tế, chúng ta đã xây dựng một ứng dụng web Flask.

**Cách chạy ứng dụng web:**
```bash
python app.py
```

Sau đó truy cập: `http://localhost:5000`


In [ ]:
# Lưu mô hình và các file cần thiết
os.makedirs('models', exist_ok=True)
os.makedirs('static', exist_ok=True)

# Lưu mô hình
joblib.dump(best_model, 'models/customer_satisfaction_model.pkl')
print("✅ Đã lưu mô hình: models/customer_satisfaction_model.pkl")

# Lưu scaler
joblib.dump(scaler, 'models/scaler.pkl')
print("✅ Đã lưu scaler: models/scaler.pkl")

# Lưu feature names
with open('models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
print("✅ Đã lưu feature names: models/feature_names.json")

# Lưu thông tin đánh giá
evaluation_info = {
    'best_model': best_model_name,
    'metrics': {
        'accuracy': float(results[best_model_name]['accuracy']),
        'precision': float(results[best_model_name]['precision']),
        'recall': float(results[best_model_name]['recall']),
        'f1': float(results[best_model_name]['f1']),
        'auc': float(results[best_model_name]['auc'])
    },
    'all_models': {name: {
        'accuracy': float(results[name]['accuracy']),
        'f1': float(results[name]['f1'])
    } for name in results}
}

with open('models/evaluation_info.json', 'w') as f:
    json.dump(evaluation_info, f, indent=2)
print("✅ Đã lưu evaluation info: models/evaluation_info.json")

print("\n🎉 Hoàn thành quy trình CRISP-DM!")


# KẾT LUẬN

## Tóm tắt Dự án

✅ **Đã hoàn thành đầy đủ 6 phases của CRISP-DM:**
1. ✅ Business Understanding - Xác định bài toán và mục tiêu
2. ✅ Data Understanding - Thu thập và khảo sát dữ liệu
3. ✅ Data Preparation - Tiền xử lý và chuẩn bị dữ liệu
4. ✅ Modeling - Xây dựng và so sánh các mô hình
5. ✅ Evaluation - Đánh giá kết quả với các metrics
6. ✅ Deployment - Triển khai ứng dụng web

## Kết quả

- **Mô hình tốt nhất**: {best_model_name}
- **Accuracy**: {accuracy:.2%}
- **F1-Score**: {f1:.4f}

## Ứng dụng

- ✅ Ứng dụng web Flask hoàn chỉnh
- ✅ API endpoints để tích hợp
- ✅ Giao diện thân thiện với người dùng

---

**Cảm ơn bạn đã xem notebook này!**
